# Test objectives 1 and 3

Load a checkpoint from the training notebook and check two things:

1. **Clean file.** Embed a 7-character message, decode the stamped image, and require the BCH code to accept it.
2. **Light edits.** Brightness, contrast, JPEG, and border crops of 5%, 10%, and 15%. The 10% crop is the one the training augmentation targets. 15% is a stress test.

An unwatermarked cover is decoded as well. Its BCH check should fail. That is what keeps ordinary photos from being labeled as watermarked.

This notebook does not search a larger canvas. A 1000×900 snip that contains the stamp needs the finder, which is trained separately.


In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    %pip install -q torch torchvision
%pip install -q lpips opencv-python-headless pillow bchlib


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as error:
    print("Drive not mounted (%s)." % error)

CANDIDATES = [
    Path("/content/drive/MyDrive/own"),
    Path("/content/own"),
    Path.cwd(),
    Path.cwd().parent,
]
PROJECT_ROOT = None
for path in CANDIDATES:
    if (path / "stegwatermark").is_dir():
        PROJECT_ROOT = path
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Upload the own folder (the directory that contains stegowatermark/).")
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)


## Checkpoint and images

`CHECKPOINT` is the `latest.pt` written by the training notebook. `IMAGE_DIR` is a folder of a few held-out photos. Each photo is fitted to 400×400, same as training.

In [ ]:
CHECKPOINT = "/content/drive/MyDrive/watermark_checkpoints/latest.pt"
IMAGE_DIR = "/content/data/mirflickr_holdout"
MESSAGE = "Hello"
MAX_IMAGES = 8


In [ ]:
from pathlib import Path
from stegowatermark.eval import evaluate_cover, load_cover, load_encoder_decoder, stamp_to_pil

encoder, decoder, device, step = load_encoder_decoder(CHECKPOINT)
print("loaded step", step, "on", device)

folder = Path(IMAGE_DIR)
paths = [p for p in sorted(folder.rglob("*")) if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
if not paths:
    raise FileNotFoundError("no images in %s" % IMAGE_DIR)
paths = paths[:MAX_IMAGES]
print("testing", len(paths), "images")


## Results

`bch_ok` is the decision bit. Bit accuracy is there so a near-miss (a crop that flipped a few bits) is visible even when the code still repairs it.

The last row of each image is the cover **without** a watermark. `bch_ok` on that row should be false.


In [ ]:
from IPython.display import display
import pandas as pd

out_dir = Path("/content/stamped_preview")
out_dir.mkdir(parents=True, exist_ok=True)

all_rows = []
for path in paths:
    cover = load_cover(path)
    rows, stamped = evaluate_cover(encoder, decoder, cover, MESSAGE)
    stamped_path = out_dir / (path.stem + "_stamped.png")
    stamp_to_pil(stamped).save(stamped_path)
    display(stamp_to_pil(stamped).resize((200, 200)))
    for row in rows:
        row["image"] = path.name
        all_rows.append(row)

table = pd.DataFrame(all_rows)[["image", "edit", "bit_acc", "bch_ok", "text"]]
display(table)


## How to read a failure

| What failed | What it means |
|---|---|
| Clean file, BCH not ok | Training has not learned the message yet. Check `clean` accuracy in `train_log.csv`. |
| Brightness or JPEG fails, clean works | The distortion ramps were still low when training stopped, or the checkpoint is early. |
| 10% crop fails, clean works | Let the crop ramp finish (it reaches 10% about 20k steps after step 1500) and keep training. |
| Unwatermarked cover has `bch_ok` true | The code check is too loose for this checkpoint. Do not use it as an AI / not-AI flag yet. |
